# Train Your First Custom Wake Word with Nanowakeword!

Welcome to the official tutorial for **Nanowakeword**!

In this notebook, we will guide you through the entire process of training a high-performance, custom wake word model from scratch. You don't need any pre-existing data-we will download everything we need and let Nanowakeword's intelligent engine do the heavy lifting.

**Our goal:** Go from zero to a ready-to-use wake word model in just a few simple steps. Let's get started!

**Installation**

In [1]:
# @title Step 1: Install Nanowakeword
# We install the full [train] package to get all the necessary dependencies.

! pip install "nanowakeword[train] @ git+https://github.com/arcosoph/nanowakeword.git" # Or download from pypi
! pip install piper-tts

print("Installation complete!")

  Cloning https://github.com/arcosoph/nanowakeword.git to /tmp/pip-install-2fcd7ym7/nanowakeword_533c63249d404ddabec18bc7a64faaa0
  Running command git clone --filter=blob:none --quiet https://github.com/arcosoph/nanowakeword.git /tmp/pip-install-2fcd7ym7/nanowakeword_533c63249d404ddabec18bc7a64faaa0
  Resolved https://github.com/arcosoph/nanowakeword.git to commit 569cfb1f426acfd5a01e0ff01ff930f168ac2711
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 47.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.1/86.1 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.7/195.7 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Step 2: Prepare the Dataset

A great model starts with great data. For this tutorial, we will:
1.  **Download** open-source noise and Room Impulse Response (RIR) datasets.
2.  **Organize** all your project files within a clean, well-structured folder hierarchy for better clarity and maintainability.

In [ ]:
import os
import time
import requests
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# output dir
out_dir = "data/background_noise"
os.makedirs(out_dir, exist_ok=True)

session = requests.Session()

retry = Retry(
    total=5,
    backoff_factor=1,
    status_forcelist=[500, 502, 503, 504],
)

adapter = HTTPAdapter(max_retries=retry)

session.mount("http://", adapter)
session.mount("https://", adapter)

api = "https://api.github.com/repos/karolpiczak/ESC-50/contents/audio"

files = [
    f["name"]
    for f in session.get(api, timeout=30).json()
    if f["name"].endswith(".wav")
]

base = "https://raw.githubusercontent.com/karolpiczak/ESC-50/master/audio/"

# Download function
def dl(fname):
    path = os.path.join(out_dir, fname)

    if os.path.exists(path):
        return

    url = base + fname

    for attempt in range(5):
        try:
            r = session.get(url, stream=True, timeout=60)
            r.raise_for_status()

            with open(path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)

            return

        except Exception as e:
            print(f"[Retry {attempt+1}] {fname} -> {e}")
            time.sleep(2)

    print(f"[FAILED] {fname}")

# Parallel download
with ThreadPoolExecutor(max_workers=8) as ex:
    list(tqdm(ex.map(dl, files), total=len(files)))

 44%|████▍     | 442/1000 [00:14<00:18, 30.01it/s]

### Download Preprocessed Features

In [3]:
!wget https://huggingface.co/datasets/arcosoph/RACON_11h_v1/resolve/main/RACON_11h_v1.npy
!wget https://huggingface.co/datasets/arcosoph/AE29H_float32/resolve/main/AE29H_float32.npy
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy

--2026-08-31 01:09:43--  https://huggingface.co/datasets/arcosoph/RACON_11h_v1/resolve/main/RACON_11h_v1.npy
Resolving huggingface.co (huggingface.co)... 3.170.185.14, 3.170.185.33, 3.170.185.35, ...
Connecting to huggingface.co (huggingface.co)|3.170.185.14|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/69d1cbbc59ede2545f741347/c7ae9d9f0e79156a65217b27e20d6729a65c419c9dfa998d0a65447fdd46f251?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27RACON_11h_v1.npy%3B+filename%3D%22RACON_11h_v1.npy%22%3B&user_id=public&X-Xet-Cas-Uid=public&Expires=1788142183&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjlkMWNiYmM1OWVkZTI1NDVmNzQxMzQ3L2M3YWU5ZDlmMGU3OTE1NmE2NTIxN2IyN2UyMGQ2NzI5YTY1YzQxOWM5ZGZhOTk4ZDBhNjU0NDdmZGQ0NmYyNTFcXD9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSomdXNlcl9pZD1wdWJsaWMmWC1YZXQtQ2FzLVVpZD1wdWJsaWMiLCJDb25kaXRpb24iOnsiRGF0ZUxlc3NUaGFuIjp7IkVwb2NoVGlt

## Step 3: Configure and Train the Model

Now for the fun part! We will create a `config.yaml` file and then run the Nanowakeword training command.

In [15]:
# @title Step 3.1: Create the "NanoWakeWord" Configuration File

# ⚠️ FULL config Explanation: https://qutorium.arcosoph.com/post/nanowakeword-configuration-guide-pp1ia5
# 😄 ༼ つ ◕_◕ ༽つ Please read our comment & change your required before starting this section.

import yaml

config_dict = {
    # ==========================================================
    # Project & Data Paths
    # ==========================================================
    "model_name": "XiaoC_A_v1",  # ⚠️ Change this name when creating a new model (e.g., arcosoph_A_v2)
    "output_dir": "./trained_models",

    "positive_data_path": "./data/positive",
    "negative_data_path": "./data/negative",
    # "background_paths": ["./SonicWeave-v2"], # https://huggingface.co/datasets/arcosoph/SonicWeave-v2
    "background_paths": ["./data/background_noise"],
    "rir_paths": ["./data/rir"], # You can leave it blank if you want (⚠️ It is better not to use it)

    # ==========================================================
    # Model Architecture
    # ==========================================================
    "model_type": "dnn", # You can try other architectures. Fast Time `DNN` is recommended.
    "layer_size": 32, # You can increase it further to get better results (e.g., 64, 128)
    "n_blocks": 3,    # You can increase it further to get better results (e.g., 4, 6)
    "embedding_dim": 128,
    "dropout_prob": 0.3,
    "activation_function": "relu",

    # ==========================================================
    # Loss Function
    # ==========================================================
    "margin_pos": 2.0,
    "margin_neg": -2.0,
    "LOSS_BIAS": 0.65,

    "logit_reg_weight": 0.0005,
    "logit_reg_margin": 4.0,
    "logit_min_margin": 1.5,

    # ==========================================================
    # Training Settings
    # ==========================================================
    "steps": 50000, # Traning steps
    "stabilization_steps": 20000, # After how many steps will validation be activated?  (⚠️: stabilization_steps < steps)

    "optimizer_type": "adamw",
    "learning_rate_max": 0.0008,
    "lr_scheduler_type": "onecycle",
    "weight_decay": 0.01,
    "momentum": 0.9,
    "num_workers": 0,


    # ==========================================================
    # Batch Composition
    # ==========================================================
    # ⚠️ ༼ つ ◕_◕ ༽つ Every feature key used inside `feature_manifest`
    # must also be added here with a batch value.
    # Otherwise, that dataset will not be included during training.
    "batch_composition": {
        "t": 100,
        "n": 100,
        "b": 90,
        "hn": 20,
        "oww": 1000,
        "AE28H_float32": 100,
        # "Other": ...
    },


    # ==========================================================
    # Feature Manifest
    # ==========================================================
    # ⚠️ ༼ つ ◕_◕ ༽つ Please make sure all dataset paths are set correctly.
    # Dataset filenames and locations may differ on your system,
    # so always use the actual path to the corresponding file.
    "feature_manifest": {  # Multiple feature datasets can be added here with any custom name/key.
        "targets": {
            "t": "./trained_models/XiaoC_A_v1/features/positive_features.npy"
        },
        "negatives": {
            "AE28H_float32": "./AE29H_float32.npy",
            "b": "./RACON_11h_v1.npy",
            "n": "./trained_models/XiaoC_A_v1/features/negative_features.npy",
            "hn": "./trained_models/XiaoC_A_v1/features/hard_negative_features.npy",
            "oww": "./openwakeword_features_ACAV100M_2000_hrs_16bit.npy",
            # "Other": ......
        },
        "targets_val": {
            "t_v": "./trained_models/XiaoC_A_v1/features/positive_features_val.npy"
        },
        "negatives_val": {
            "bv": "RACON_11h_v1.npy"
        }
    },


    # ==========================================================
    # Synthetic Data Generation
    # ==========================================================
    "target_phrase": "hey Xiao C",  # ⚠️ your wake word

    "data_generation_tasks": [
        {
            "name": "any name like pos_data",
            "enabled": True,
            "output_dir": "data/positive",
            "num_samples": 2500,
            "file_prefix": "pos",
            "text_source": {
                "type": "fixed_phrase",
                "phrase": "hey Xiao C" # ⚠️ your wake word
            }
        },
        {
            "name": "pos_val",
            "enabled": True,
            "output_dir": "data/positive_val",
            "num_samples": 2000,
            "file_prefix": "pos",
            "text_source": {
                "type": "fixed_phrase",
                "phrase": "hey Xiao C" # ⚠️ your wake word
            }
        },
        {
            "name": "Adversarial Negatives",
            "enabled": True,
            "output_dir": "data/negative",
            "num_samples": 5000,
            "file_prefix": "neg_auto",
            "text_source": {
                "type": "auto_adversarial",
                "base_phrase": "hey Xiao C", # ⚠️ your wake word
                "include_input_words": True,
                "include_partial_phrase": True,
                "multi_word_prob": 0.5,
                "max_multi_word_len": 3
            }
        },
        {
            "name": "Phoneme Hard Negatives",
            "enabled": True,
            "output_dir": "data/negative_phoneme",
            "num_samples": 3000,
            "file_prefix": "neg_ph",
            "text_source": {
                "type": "phoneme_adversarial",
                "base_phrase": "hey Xiao C", # ⚠️ your wake word
                "min_distance": 0.3
            }
        },
        {
            "name": "Custom Negatives",
            "enabled": True,
            "output_dir": "data/negative",
            "num_samples": 50,
            "file_prefix": "neg_custom",
            "text_source": {
                "type": "from_list",
                "phrases": [           # ⚠️ Use your own negative words if necessary
                    "arcosoph eloo",
                    "arcosop",
                    "hie arcosoph",
                    "arkhoshap",
                    "yarkasop"
                ],
                "repeat_each": 10
            }
        }
    ],


    # ==========================================================
    # Augmentation
    # ==========================================================
    "augmentation_batch_size": 16,
    "feature_gen_cpu_ratio": 1.0,

    "augmentation_settings": {
        "gain_prob": 1.0,
        "max_gain_in_db": 2.0,
        "max_pitch_semitones": 1.0,
        "max_snr_in_db": 35.0,
        "min_gain_in_db": -2.0,
        "min_pitch_semitones": -1.0,
        "min_snr_in_db": 15.0,
        "pitch_prob": 0.3,
        "rir_prob": 0.0
    },

    # ==========================================================
    # Feature Generation
    # ==========================================================
    "feature_generation_manifest": {   # This can be slow generation on the Colab or Gpu
        "jast_a_name_like_pos_feature": {
            "input_audio_dirs": ["./data/positive"],
            "output_filename": "positive_features.npy",
            "use_background_noise": True,
            "use_rir": False,
            "augmentation_rounds": 10
        },
        "pos_val_feature": {
            "input_audio_dirs": ["./data/positive_val"],
            "output_filename": "positive_features_val.npy",
            "use_background_noise": True,
            "use_rir": False,
            "augmentation_rounds": 10
        },
        "negatives_1": {
            "input_audio_dirs": ["./data/negative"],
            "output_filename": "negative_features.npy",
            "use_background_noise": True,
            "use_rir": False,
            "augmentation_rounds": 10
        },
        "negatives_2": {
            "input_audio_dirs": ["./data/negative_phoneme"],
            "output_filename": "hard_negative_features.npy",
            "use_background_noise": True,
            "use_rir": False,
            "augmentation_rounds": 1
        }
    },

    # "clip_length_samples": # ⚠️⚠️ ༼ つ ◕_◕ ༽つ  It is very good to use the clip_length_samples parameter (e.g., 32000, etc Which is for you)
    "background_paths_duplication_rate": [1],

    # ==========================================================
    # Validation
    # ==========================================================
    "val_miss_weight": 4.0,
    "val_fp_weight": 1.0,
    "validation_batch_size": 256,
    "validation_smoothing_window": 3,
    "val_early_stopping_patience": 6000,

    # ==========================================================
    # Curriculum Learning
    # ==========================================================
    "hardness_ema_alpha": 0.05,
    "hardness_floor": 0.05,
    "hardness_reset_interval": 5000,
    "hardness_reset_decay": 0.5,
    "checkpoint_averaging_top_k": 5,

    # ==========================================================
    # Checkpointing & Debug
    # ==========================================================
    "checkpointing": {
        "enabled": True,
        "interval_steps": 1000,
        "limit": 2
    },

    "early_stopping_patience": 0,
    "min_delta": 0.0001,
    "ema_alpha": 0.01,

    "onnx_opset_version": 17,
    "show_training_summary": True,
    "debug_mode": True,

    # ==========================================================
    # Pipeline Control
    # ==========================================================
    "transform_clips": False, #⚠️ Set to False after generating clips to avoid unnecessary reprocessing
    "generate_clips": False,  #⚠️ Set to False after generating clips to avoid unnecessary reprocessing
    "train_model": True,
    "overwrite": False
}

config_path = "./config.yaml"
with open(config_path, "w") as f:
    yaml.dump(config_dict, f, default_flow_style=False, sort_keys=False)

print("⚠️[NOTE] Set `transform_clips: False` and `generate_clips: False`, then re-run this cell to create a new model, unless regeneration of audio data or feature files (.npy) is required.\n")
print("config.yaml written successfully")

⚠️[NOTE] Set `transform_clips: False` and `generate_clips: False`, then re-run this cell to create a new model, unless regeneration of audio data or feature files (.npy) is required.

config.yaml written successfully


**Run Training!**

In [16]:
# @title Step 3.2: Run the Magic Command! 🚀
# This command will do everything: augment data, extract features, and train the model.
# It might take some time depending on the hardware (especially on a CPU).

from nanowakeword.trainer import train

args_list = [
    '--config_path', f'{config_path}',
]

print("Starting NanoWakeWord training...")

try:
    train(args_list)
    print("\n\nCONGRATULATIONS! (✿◕‿◕✿)")
    print("Your custom wake word model has been successfully trained!")

except Exception as e:
    print(f"\nAn error occurred during training: {e}")

Starting NanoWakeWord training...


  _   _               __          __   _     __          __           _ 
 | \ | |              \ \        / /  | |    \ \        / /          | |
 |  \| | __ _ _ __   __\ \  /\  / /_ _| | ____\ \  /\  / /__  _ __ __| |
 | . ` |/ _` | '_ \ / _ \ \/  \/ / _` | |/ / _ \ \/  \/ / _ \| '__/ _` |
 | |\  | (_| | | | | (_) \  /\  / (_| |   <  __/\  /\  / (_) | | | (_| |
 |_| \_|\__,_|_| |_|\___/ \/  \/ \__,_|_|\_\___| \/  \/ \___/|_|  \__,_|

INFO: Project assets: /content/trained_models/XiaoC_A_v1

INFO: Scanning dataset and building index pools...

INFO: Dataset Successfully Initialized [Embedding (Memmap)] | Sources: 6 | Total Samples: 6205960

...

INFO: Checkpointing is ENABLED. A checkpoint will be saved every 1000 steps.

INFO: Early stopping is DISABLED; the training process will run for the full 50000 steps.

                                       Effective Training Configuration                                       
┌──────────────────────────────────────────┬─────────────────────────────────────────────────────────────────┐
│ Parameter                                │ Value                                                           │
├──────────────────────────────────────────┼─────────────────────────────────────────────────────────────────┤
│ activation_function                      │ relu                                                            │
│ batch_composition.AE28H_float32          │ 100                                                             │
│ batch_composition.b                      │ 90                                                              │
│ batch_composition.hn                     │ 20                                                              │
│ batch_composition.n                      │ 100                                                             │
│ batch_composition.oww                    │ 1000                                                            │
│ batch_composition.t                      │ 100                                                             │
│ checkpoint_averaging_top_k               │ 5                                                               │
│ checkpointing.enabled                    │ True                                                            │
│ checkpointing.interval_steps             │ 1000                                                            │
│ checkpointing.limit                      │ 2                                                               │
│ debug_mode                               │ True                                                            │
│ dropout_prob                             │ 0.3                                                             │
│ early_stopping_patience                  │ 0                                                               │
│ ema_alpha                                │ 0.01                                                            │
│ embedding_dim                            │ 128                                                             │
│ feature_manifest.negatives.AE28H_float32 │ ./AE29H_float32.npy                                             │
│ feature_manifest.negatives.b             │ ./RACON_11h_v1.npy                                              │
│ feature_manifest.negatives.hn            │ ./trained_models/XiaoC_A_v1/features/hard_negative_features.npy │
│ feature_manifest.negatives.n             │ ./trained_models/XiaoC_A_v1/features/negative_features.npy      │
│ feature_manifest.negatives.oww           │ ./openwakeword_features_ACAV100M_2000_hrs_16bit.npy             │
│ feature_manifest.negatives_val.bv        │ RACON_11h_v1.npy                                                │
│ feature_manifest.targets.t               │ ./trained_models/XiaoC_A_v1/features/positive_features.npy      │
│ feature_manifest.targets_val.t_v         │ ./trained_models/XiaoC_A_v1/features/positive_features_val.npy  │
│ layer_size                               │ 32                                                              │
│ learning_rate_max                        │ 0.0008                                                          │
│ lr_scheduler_type                        │ onecycle                                                        │
│ min_delta                                │ 0.0001                                                          │
│ mode                                     │ embedding                                                       │
│ model_name                               │ XiaoC_A_v1                                                      │
│ model_type                               │ dnn                                                             │
│ momentum                                 │ 0.9                                                             │
│ n_

Training:   0%|          | 5/50000 [00:28<78:06:11,  5.62s/it]


KeyboardInterrupt: 

## What's Next?

You have successfully trained your own custom wake word model!

You can now download the `.onnx` or `.pt` file from the `trained_models` directory (check the file browser on the left) and use it in your own applications.

For more advanced topics, such as using your own datasets or fine-tuning the configuration, please check out our full documentation on **[GitHub](https://github.com/arcosoph/nanowakeword)**.

---
## Step 4: Save Your Model to Google Drive

The final step is to save your trained model and performance graph to a safe and accessible place. Instead of a slow direct download, we will save the files directly to your Google Drive. This process is almost instantaneous.

Run the cells below to:
1.  Connect your Google Drive account.
2.  Copy all the trained files into a new folder named `nanowakeword_models` in your Drive.

In [ ]:
# @title Step 4.1: Connect to Google Drive
# This will ask for your permission to access your Google Drive.

from google.colab import drive

try:
    drive.mount('/content/drive')
    print("\nGoogle Drive connected successfully!")
except Exception as e:
    print(f"An error occurred while connecting to Google Drive: {e}")

In [ ]:
# @title Step 4.2: Copy Final Model and Artifacts to Google Drive 📂

import os
import shutil

# --- Configuration ---
# Get model_name and output_dir from the config_dict defined earlier
model_name = config_dict.get("model_name", "my_model")
output_dir = config_dict.get("output_dir", "./trained_models")

# --- Source and Destination Paths ---
# The source project directory containing all generated files
source_project_dir = os.path.join(output_dir, model_name)

# The destination folder in your Google Drive
drive_destination_dir = f"drive/MyDrive/nanowakeword_models/{model_name}"

# --- Start Copy Process ---
print("Starting the process to copy trained files to Google Drive...")

# Check if the source directory exists
if not os.path.exists(source_project_dir):
    print(f"\nERROR: Source directory not found at '{source_project_dir}'")
    print("This indicates that the training process did not create the expected output folder.")
    print("Please ensure the training step completed successfully before running this cell.")
else:
    # If an old folder exists in Drive, remove it to ensure a clean copy
    if os.path.exists(drive_destination_dir):
        print(f"🔄 Found an existing folder in Drive. Removing it for a fresh copy: '{drive_destination_dir}'")
        shutil.rmtree(drive_destination_dir)

    # --- Copy the entire project folder ---
    # This is much simpler and more reliable than copying individual files.
    # It preserves the professional directory structure.
    try:
        shutil.copytree(source_project_dir, drive_destination_dir)

        print("\n" + "="*50)
        print("✅ SUCCESS! All files have been saved to your Google Drive.")
        print("="*50)
        print(f"\nYour complete project, including the model and performance graphs, can be found in:")
        print(f"➡️ '{drive_destination_dir}'")

        # Optional: List the contents of the new folder in Drive for verification
        print("\nContents of the saved folder:")
        for root, dirs, files in os.walk(drive_destination_dir):
            level = root.replace(drive_destination_dir, '').count(os.sep)
            indent = ' ' * 4 * (level)
            print(f"{indent}{os.path.basename(root)}/")
            sub_indent = ' ' * 4 * (level + 1)
            for f in files:
                print(f"{sub_indent}{f}")

    except Exception as e:
        print(f"\nERROR: An unexpected error occurred during the copy process.")
        print(f"Details: {e}")

<p style="font-size:18px; font-weight:600;">
  💙 If you find this helpful, please support us at
  <a href="https://arcosoph.com" style="text-decoration:none;">
    <span style="color:#fefefe;">A</span>
    <span style="color:#2cab4e;">r</span>
    <span style="color:#029adb;">c</span>
    <span style="color:#821720;">o</span>
    <span style="color:#f9e91b;">s</span>
    <span style="color:#821720;">o</span>
    <span style="color:#fefefe;">p</span>
    <span style="color:#f9e91b;">h</span>
  </a>  or give our
  <a href="https://github.com/arcosoph/NanoWakeWord" style="color:#007BFF; font-weight:bold; text-decoration:none;">
    repository
  </a> a ⭐
</p>